# Translation and Summarization with Seq2Seq

<a href="https://colab.research.google.com/github/HassanAlgoz/dl/blob/main/modules/pipelines/02-pipelines/05_seq2seq.ipynb" target="_blank">
  <img src="https://raw.githubusercontent.com/HassanAlgoz/dl/main/assets/Open%20in%20Colab-F9AB00.svg" alt="Open in Colab" height="50"/>
</a>

Encoder-decoder (**seq2seq**) models transform one sequence into another — ideal for **translation** and **summarization** of multilingual customer reviews.


**Goal:** Apply seq2seq models to translation and summarization.

**Install:** `%pip install -qqq transformers sentencepiece`


In [1]:
# --- Setup: Clone repo & cd into correct folder (Colab only) ---
import os
import sys
import subprocess

if "google.colab" in sys.modules:
    repo_url = "https://github.com/HassanAlgoz/dl.git"
    lab_folder = "dl/modules/pipelines/02-pipelines"

    # Only clone if the folder doesn't exist
    if not os.path.exists(lab_folder):
        subprocess.run(["git", "clone", repo_url])

    # Change working directory to the lab folder
    os.chdir(lab_folder)


## Imports


In [2]:
%pip install -qqq transformers==4.45.2 sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 62.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [3]:
import numpy as np
import pandas as pd

from transformers import pipeline

## NLP Scenario

You are an analyst for a marketing company that just launched a new product suite of mobile devices. You have data from product reviews of the new product and need to translate non-English reviews and summarize long reviews.

1. Translate non-English reviews to English
2. Summarize long reviews for quick analysis

#### Sample Reviews

Spanish review:
```
¡Me encanta TechWave X1! Ha hecho que mis tareas diarias en España sean mucho más fáciles y eficientes.
¡Lo recomiendo encarecidamente!
```

Long Review:
```
I've been using the TechWave X1 mobile phone for the past three months, and I want to share my comprehensive experience. The device initially caught my attention with its sleek design and promised features, but I've discovered both strengths and weaknesses during daily use.
On the positive side, the battery life is exceptional - I consistently get through a full day of heavy use with around 30% charge remaining. The 10-hour screen time is impressive, especially considering I frequently use power-hungry apps. The camera system is another standout feature. In good lighting, it captures stunning photos with natural colors and excellent detail. Even in low-light conditions, the night mode produces surprisingly clear and vibrant images.
However, there are some drawbacks worth mentioning. While the interface is generally smooth, I've noticed occasional lag when switching between resource-intensive apps. The facial recognition can be inconsistent in dim lighting, though the fingerprint sensor works reliably. The built-in speakers, while adequate for calls, don't provide the best audio quality for media consumption.
The waterproof design has already proved its worth - the phone survived an unexpected rainfall during an outdoor photoshoot. The fast charging capability is also a lifesaver, getting the phone from 0 to 50% in just 25 minutes. The 5G connectivity is solid in my area, providing consistently fast data speeds.
```


In [4]:
# data setup
spanish_review= """¡Me encanta TechWave X1! Ha hecho que mis tareas diarias en España sean mucho más fáciles y eficientes.
¡Lo recomiendo encarecidamente!"""

long_review = """ I've been using the TechWave X1 mobile phone for the past three months, and I want to share my comprehensive experience. The device initially caught my attention with its sleek design and promised features, but I've discovered both strengths and weaknesses during daily use.
On the positive side, the battery life is exceptional - I consistently get through a full day of heavy use with around 30% charge remaining. The 10-hour screen time is impressive, especially considering I frequently use power-hungry apps. The camera system is another standout feature. In good lighting, it captures stunning photos with natural colors and excellent detail. Even in low-light conditions, the night mode produces surprisingly clear and vibrant images.
However, there are some drawbacks worth mentioning. While the interface is generally smooth, I've noticed occasional lag when switching between resource-intensive apps. The facial recognition can be inconsistent in dim lighting, though the fingerprint sensor works reliably. The built-in speakers, while adequate for calls, don't provide the best audio quality for media consumption.
The waterproof design has already proved its worth - the phone survived an unexpected rainfall during an outdoor photoshoot. The fast charging capability is also a lifesaver, getting the phone from 0 to 50% in just 25 minutes. The 5G connectivity is solid in my area, providing consistently fast data speeds.
"""

## Translation Pipeline

The translation pipeline can be initialized with:
```python
pipeline("translation")
```

For specific language pairs, use models like:
- `Helsinki-NLP/opus-mt-es-en` for Spanish to English
- `Helsinki-NLP/opus-mt-en-es` for English to Spanish
  
Explore available models here and languages [here](https://huggingface.co/models?sort=trending&search=Helsinki-NLP)

### Key parameters

- `max_length`: Maximum length of generated translation
- `num_beams`: Number of beams for beam search (higher = more thorough but slower)
- `do_sample`: Whether to use sampling instead of beam search


In [5]:
# initialize translation pipeline
translator = pipeline(
    task="translation",
    model="Helsinki-NLP/opus-mt-es-en"
  )


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [6]:
# translate Spanish review to English
translation = translator(spanish_review, max_length=200)
print(translation[0]['translation_text'])

I love TechWave X1! It has made my daily tasks in Spain much easier and more efficient. I highly recommend it!


## Summarization Pipeline

The summarization pipeline can be initialized with:
```python
pipeline("summarization")
```

### Key parameters

- `max_length`: Maximum length of generated summary
- `min_length`: Minimum length of generated summary
- `do_sample`: Whether to use sampling for generation
- `num_beams`: Number of beams for beam search


In [ ]:
# initialize summarization pipeline
summarizer = pipeline(
    task="summarization",
    model="sshleifer/distilbart-cnn-12-6" # (default model)
  )

# summarize review
summary = summarizer(long_review,
                    max_length=100,
                    min_length=30)


No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

In [12]:
print(summary[0]['summary_text'])

 The waterproof design has already proved its worth - the phone survived an unexpected rainfall during an outdoor photoshoot . Fast charging capability is also a lifesaver, getting the phone from 0 to 50% in just 25 minutes . While the interface is generally smooth, I've noticed occasional lag .


## Conclusion
- Seq2Seq models are powerful for tasks that transform one sequence to another
- Translation and summarization can be combined for international products
- Different models and parameters can affect output quality
- Consider the trade-offs between speed and quality when selecting models
- Real-world applications often require combining multiple models
